# Evaluation on Vezilka LLM using BBEH benchmark for reasoning tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.0 MB/s eta 0:00:00


In [2]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model_name = "finki-ukim/VezilkaLLM"

In [5]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model= AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/934 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/444 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [7]:
def load_json(path):
  with open(path,"r") as f:
    data = json.load(f)

    df = pd.DataFrame(data['examples'])
    return df

In [8]:
def get_prompt(instruction,questions):
  prompts=[]
  for i in range(len(questions)):
    question = questions[i].split("Reply")[0]
    prompt = f"{instruction}\n\n{question}\n\nAnswer: "
    prompts.append(prompt)
  return prompts

In [9]:
def training(prompts,num_tokens=1):
  predictions=[]
  for i in range(len(prompts)):
    input_tokens = tokenizer(prompts[i],return_tensors="pt").to("cuda:0")
    output_tokens = model.generate(**input_tokens,max_new_tokens=num_tokens,
                                   do_sample=False,
                                   pad_token_id=tokenizer.eos_token_id,
                                   eos_token_id=tokenizer.eos_token_id)

    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True).strip()
    pred_answer = response.split("Answer:")[1]
    print(f"Iteration {i}\n")
    print(f"Response: {pred_answer}")
    predictions.append(pred_answer)

  return predictions

## Task Casual Understanding

In [10]:
dataset = load_json("/content/task_casual_understanding.json")

In [11]:
dataset

,input,target
0,Question: Lauren and Jane work for the same co...,Yes
1,Question: Lauren and Jane work for the same co...,Yes
2,Question: The receptionist in the philosophy d...,Yes
3,Question: The receptionist in the philosophy d...,No
4,Question: Claire's parents bought her an old c...,Yes
...,...,...
195,Question: A test tube contains a mixture. Sam ...,Yes
196,Question: Reagent X and Y are both added to a ...,No
197,Question: There is a test tube on the left and...,Yes
198,Question: Sam is heating Reagent X in a test t...,Yes


In [12]:
questions = dataset['input'].values.tolist()

In [13]:
true_answers = dataset['target'].values.tolist()

In [14]:
instruction = f"""Instruction: You are an AI assistant for casual understanding. Read the story and answer with exactly one word:
Yes (if you think majority of people will answer yes), No (if you think majority of people will answer no), or Ambiguous(If you think that majority of people do not know the answer) . Do not add explanations.
"""

In [15]:
prompts = get_prompt(instruction,questions)

In [16]:
prompts[0]

"Instruction: You are an AI assistant for casual understanding. Read the story and answer with exactly one word:\nYes (if you think majority of people will answer yes), No (if you think majority of people will answer no), or Ambiguous(If you think that majority of people do not know the answer) . Do not add explanations.\n\n\nQuestion: Lauren and Jane work for the same company. They each need to use a computer for work sometimes. Unfortunately, the computer isn't very powerful. If two people are logged on at the same time, it usually crashes. So the company decided to institute an official policy. It declared that Lauren would be the only one permitted to use the computer in the mornings and that Jane would be the only one permitted to use the computer in the afternoons. As expected, Lauren logged on the computer the next day at 9:00 am. But Jane decided to disobey the official policy. She also logged on at 9:00 am. The computer crashed immediately. Did Jane cause the computer to crash

In [19]:
predictions = training(prompts,num_tokens=5)

Iteration 0

Response:  
No
Explanation:
Iteration 1

Response:  
No
Explanation:
Iteration 2

Response:  
No
Explanation:
Iteration 3

Response:  
No
Explanation:
Iteration 4

Response:  
No
Explanation:
Iteration 5

Response:  
No
Explanation:
Iteration 6

Response:  
No
Explanation:
Iteration 7

Response:  
No
Explanation:
Iteration 8

Response:  
No
Explanation:
Iteration 9

Response:  
No
Explanation:
Iteration 10

Response:  
No
Explanation:
Iteration 11

Response:  
No
Explanation:
Iteration 12

Response:  
No
Explanation:
Iteration 13

Response:  
No
Explanation:
Iteration 14

Response:  
No
Explanation:
Iteration 15

Response:  
Ambiguous
Explanation
Iteration 16

Response:  
Ambiguous
Explanation
Iteration 17

Response:  
No
Explanation:
Iteration 18

Response:  
No
Explanation:
Iteration 19

Response:  
No
Explanation:
Iteration 20

Response:  
No
Explanation:
Iteration 21

Response:  
No
Explanation:
Iteration 22

Response:  
No
Explanation:
Iteration 23

Response:  
No
Exp

In [20]:
predictions[:10]

[' \nNo\nExplanation:',
 ' \nNo\nExplanation:',
 ' \nNo\nExplanation:',
 ' \nNo\nExplanation:',
 ' \nNo\nExplanation:',
 ' \nNo\nExplanation:',
 ' \nNo\nExplanation:',
 ' \nNo\nExplanation:',
 ' \nNo\nExplanation:',
 ' \nNo\nExplanation:']

In [21]:
predictions = [elem.split("Explanation:")[0].strip() for elem in predictions]
predictions[:10]

['No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No']

In [23]:
predictions

['No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Ambiguous\nExplanation',
 'Ambiguous\nExplanation',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Bill did not cause',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Ambiguous\nExplanation',
 'Ambiguous\nExplanation',
 'No',
 'No',
 'No',
 'No',
 'No',
 

In [24]:
predictions = [elem.split("\n")[0] for elem in predictions]
predictions

['No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Ambiguous',
 'Ambiguous',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Bill did not cause',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Ambiguous',
 'Ambiguous',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'No',
 'Am

In [25]:
predictions = [elem if elem=="Yes" or elem=="No" else "Ambiguous" for elem in predictions]
predictions[:10]

['No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No']

In [26]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

   Ambiguous       0.27      0.07      0.11        45
          No       0.40      0.94      0.56        80
         Yes       0.00      0.00      0.00        75

    accuracy                           0.39       200
   macro avg       0.22      0.33      0.22       200
weighted avg       0.22      0.39      0.25       200



## Task Movie Recommendation

In [27]:
dataset = load_json("/content/task_movie_recommendation.json")

In [28]:
dataset

,input,target
0,Which option has more similar movies in terms ...,(E)
1,Which option has more similar movies in terms ...,(C)
2,Which option has more similar movies in terms ...,(C)
3,Which option has more similar movies in terms ...,(A)
4,Which option has more similar movies in terms ...,(I)
...,...,...
195,Which option has more similar movies in terms ...,(E)
196,Which option has more similar movies in terms ...,(E)
197,Which option has more similar movies in terms ...,(E)
198,Which option has more similar movies in terms ...,(A)


In [29]:
dataset["target"].value_counts()

,count
target,
(A),26
(G),22
(H),21
(E),21
(J),20
(D),20
(C),18
(F),18
(B),17


In [30]:
questions = dataset['input'].values.tolist()

In [31]:
true_answers = dataset['target'].values.tolist()

In [32]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'C', 'A', 'I', 'D', 'B', 'F', 'D', 'H']

In [33]:
instruction=f"""Instruction: You are an AI assistant that determines which group of movies is most similar.
Answer only with the one of the letters (**A**, **B**, **C**, **D**, **E**, **F**, **G**, **H**, **I**, or **J**) of the group that is most similar and then stop immediately. Do not write anything else.
Final output format: A,B,C,D,E,F,G,H,I or J
"""

In [34]:
prompts = get_prompt(instruction,questions)

In [35]:
prompts[1]

'Instruction: You are an AI assistant that determines which group of movies is most similar.\nAnswer only with the one of the letters (**A**, **B**, **C**, **D**, **E**, **F**, **G**, **H**, **I**, or **J**) of the group that is most similar and then stop immediately. Do not write anything else.\nFinal output format: A,B,C,D,E,F,G,H,I or J\n\n\nWhich option has more similar movies in terms of whether a group of people will like them or not?\nOptions:\n(A) Street Fighter II The Animated Movie, The Sixth Sense, The Matrix, The Shawshank Redemption, Terminator 2 Judgment Day\n(B) The Boy Who Could Fly, The Sheltering Sky, Forrest Gump, Terminator 2 Judgment Day, The Matrix\n(C) Forrest Gump, The Shawshank Redemption, Terminator 2 Judgment Day, The Matrix, The Sixth Sense\n(D) The Boy Who Could Fly, The Sheltering Sky, Street Fighter II The Animated Movie, The Sixth Sense, Terminator 2 Judgment Day\n(E) The Matrix, The Boy Who Could Fly, Forrest Gump, The Shawshank Redemption, The Sixth Se

In [42]:
predictions = training(prompts,num_tokens=3)

Iteration 0

Response:  
The option
Iteration 1

Response:  
A
Iteration 2

Response:  
A
Iteration 3

Response:  
A,
Iteration 4

Response:  
A
Iteration 5

Response:  
A
Iteration 6

Response:  12 Angry
Iteration 7

Response:  
A
Iteration 8

Response:  
A
Iteration 9

Response:  
A
Iteration 10

Response:  
The group
Iteration 11

Response:  
A
Iteration 12

Response:  
A
Iteration 13

Response:  
The group
Iteration 14

Response:  
The option
Iteration 15

Response:  
A
Iteration 16

Response:  
The Art
Iteration 17

Response:  
A
Iteration 18

Response:  
A
Iteration 19

Response:  1.
Iteration 20

Response:  
A
Iteration 21

Response:  
A
Iteration 22

Response:  
A
Iteration 23

Response:  
A
Iteration 24

Response:  
The option
Iteration 25

Response:  
A
Iteration 26

Response:  
A
Iteration 27

Response:  
A
Iteration 28

Response:  
A
Iteration 29

Response:  
A
Iteration 30

Response:  
The option
Iteration 31

Response:  
A
Iteration 32

Response:  
A
Iteration 33

Respons

In [43]:
predictions

[' \nThe option',
 ' \nA',
 ' \nA',
 ' \nA,',
 ' \nA',
 ' \nA',
 ' 12 Angry',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe group',
 ' \nA',
 ' \nA',
 ' \nThe group',
 ' \nThe option',
 ' \nA',
 ' \nThe Art',
 ' \nA',
 ' \nA',
 ' 1.',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe option',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe option',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe option',
 ' \nA,',
 ' \nA',
 ' \nA,',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe option',
 ' \nGint',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nH',
 ' \nG',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nG',
 ' \nA',
 ' \nThe option',
 ' \nA',
 ' \nA',
 ' \nI',
 ' \nA',
 ' \nThe option',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe Shaw',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe group',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe option',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nThe option',
 ' \nA,',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nA',
 ' \nTh

In [45]:
predictions = [elem.strip() for elem in predictions]
predictions

['The option',
 'A',
 'A',
 'A,',
 'A',
 'A',
 '12 Angry',
 'A',
 'A',
 'A',
 'The group',
 'A',
 'A',
 'The group',
 'The option',
 'A',
 'The Art',
 'A',
 'A',
 '1.',
 'A',
 'A',
 'A',
 'A',
 'The option',
 'A',
 'A',
 'A',
 'A',
 'A',
 'The option',
 'A',
 'A',
 'A',
 'A',
 'A',
 'The option',
 'A,',
 'A',
 'A,',
 'A',
 'A',
 'A',
 'A',
 'A',
 'The option',
 'Gint',
 'A',
 'A',
 'A',
 'H',
 'G',
 'A',
 'A',
 'A',
 'A',
 'G',
 'A',
 'The option',
 'A',
 'A',
 'I',
 'A',
 'The option',
 'A',
 'A',
 'A',
 'A',
 'A',
 'The Shaw',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'The group',
 'A',
 'A',
 'A',
 'The option',
 'A',
 'A',
 'A',
 'The option',
 'A,',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'The option',
 'A',
 'A',
 'A',
 'The Good',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'The option',
 'The option',
 'The option',
 'A,',
 'A',
 'A',
 'A',
 'The group',
 'G',
 'The group',
 'A',
 'I',
 'A,',
 'A',
 '1.',
 '(A',
 'The group',
 'A',
 'The group',
 'The optio

In [46]:
predictions = [elem if elem  in list("ABCDEFGHIJ") else "Unknown" for elem in predictions]
predictions

['Unknown',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'Unknown',
 'Unknown',
 'A',
 'Unknown',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'Unknown',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'Unknown',
 'A',
 'A',
 'A',
 'H',
 'G',
 'A',
 'A',
 'A',
 'A',
 'G',
 'A',
 'Unknown',
 'A',
 'A',
 'I',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'Unknown',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'Unknown',
 'Unknown',
 'Unknown',
 'A',
 'A',
 'A',
 'Unknown',
 'G',
 'Unknown',
 'A',
 'I',
 'Unknown',
 'A',
 'Unknown',
 'Unknown',
 'Unknown',
 'A',
 'Unknown',
 'Unknown',
 'A',
 'Un

In [47]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.16      0.81      0.26        26
           B       0.00      0.00      0.00        17
           C       0.00      0.00      0.00        18
           D       0.00      0.00      0.00        20
           E       0.00      0.00      0.00        21
           F       0.00      0.00      0.00        18
           G       0.50      0.14      0.21        22
           H       0.00      0.00      0.00        21
           I       0.00      0.00      0.00        17
           J       0.00      0.00      0.00        20
     Unknown       0.00      0.00      0.00         0

    accuracy                           0.12       200
   macro avg       0.06      0.09      0.04       200
weighted avg       0.08      0.12      0.06       200



## Task Disambiguation QA

In [48]:
dataset = load_json("/content/task_disambiguationQA.json")

In [49]:
dataset['target'].value_counts()

,count
target,
(B),25
(D),24
(C),23
(A),22
(E),17
(F),7
(G),1
(H),1


In [50]:
questions = dataset['input'].values.tolist()

In [51]:
true_answers = dataset['target'].values.tolist()

In [52]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['A', 'D', 'E', 'B', 'A', 'B', 'B', 'A', 'C', 'D']

In [53]:
instruction = f"""You are an AI assistant solving a pronoun disambiguation task.

For each sentence:
- Identify what the pronoun(s) most likely refer to based on common sense.
- If more than one interpretation is equally plausible, select “Ambiguous”.

Rules:
- Do not assume information that is not stated.
- Do not infer intent beyond what is commonly assumed.
- Choose exactly one answer from (A)–(H) based on available letters.

Final answer format:
(A), (B), (C), (D), (E), (H), (G), or (F)
"""

In [54]:
prompts = get_prompt(instruction,questions)

In [55]:
prompts[1]

'You are an AI assistant solving a pronoun disambiguation task.\n\nFor each sentence:\n- Identify what the pronoun(s) most likely refer to based on common sense.\n- If more than one interpretation is equally plausible, select “Ambiguous”.\n\nRules:\n- Do not assume information that is not stated.\n- Do not infer intent beyond what is commonly assumed.\n- Choose exactly one answer from (A)–(H) based on available letters.\n\nFinal answer format:\n(A), (B), (C), (D), (E), (H), (G), or (F)\n\n\nHere is a sentence with pronoun(s) whose antecedent(s) can either be derived from the context or is ambiguous.\nSentence: The designer called the janitor and asked him to clean the room as soon as possible, but he went to fetch a cup of coffee first and then returned to his desk.\nWhich of the following options correctly explains the antecedent(s) of the pronoun(s)?\n(A) Designer is cleaning the room as soon as possible and got coffee.\n(B) Janitor is cleaning the room as soon as possible and got co

In [56]:
predictions = training(prompts,num_tokens=3)

Iteration 0

Response:  
(C
Iteration 1

Response:  
(A
Iteration 2

Response:  
(C
Iteration 3

Response:  
(A
Iteration 4

Response:  
(C
Iteration 5

Response:  
(B
Iteration 6

Response:  
(C
Iteration 7

Response:  
(A
Iteration 8

Response:  
(B
Iteration 9

Response:  
(C
Iteration 10

Response:  
(C
Iteration 11

Response:  
(A
Iteration 12

Response:  
(C
Iteration 13

Response:  
(C
Iteration 14

Response:  
(C
Iteration 15

Response:  
(C
Iteration 16

Response:  
(A
Iteration 17

Response:  
(C
Iteration 18

Response:  
(A
Iteration 19

Response:  
(B
Iteration 20

Response:  
(B
Iteration 21

Response:  
(A
Iteration 22

Response:  
(B
Iteration 23

Response:  
(C
Iteration 24

Response:  
(C
Iteration 25

Response:  
(A
Iteration 26

Response:  
(A
Iteration 27

Response:  
(B
Iteration 28

Response:  
(A
Iteration 29

Response:  
(A
Iteration 30

Response:  
(A
Iteration 31

Response:  
(A
Iteration 32

Response:  
(A
Iteration 33

Response:  
(C
Iteration 34

Response: 

In [57]:
predictions

[' \n(C',
 ' \n(A',
 ' \n(C',
 ' \n(A',
 ' \n(C',
 ' \n(B',
 ' \n(C',
 ' \n(A',
 ' \n(B',
 ' \n(C',
 ' \n(C',
 ' \n(A',
 ' \n(C',
 ' \n(C',
 ' \n(C',
 ' \n(C',
 ' \n(A',
 ' \n(C',
 ' \n(A',
 ' \n(B',
 ' \n(B',
 ' \n(A',
 ' \n(B',
 ' \n(C',
 ' \n(C',
 ' \n(A',
 ' \n(A',
 ' \n(B',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(C',
 ' \n(B',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(C',
 ' \n(C',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(C',
 ' \n(B',
 ' \n(A',
 ' \n(A',
 ' \n(C',
 ' \n(E',
 ' \n(A',
 ' \n(C',
 ' \n(B',
 ' \n(C',
 ' \n(C',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(E',
 ' \n(C',
 ' \n(C',
 ' \n(A',
 ' \n(C',
 ' \n(C',
 ' \n(A',
 ' \n(A',
 ' \n(C',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(C',
 ' \n(A',
 ' \n(C',
 ' \n(A',
 ' \n(A',
 ' \n(C',
 ' \n(C',
 ' \n(C',
 ' \n(A',
 ' \n(C',
 ' \n(C',
 ' \n(C',
 ' \n(C',
 ' \n(C',
 ' \n(B',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(A',
 ' \n(C',
 ' \n(A',
 ' \n(C',
 ' \n(C',
 ' \n(B',
 ' \n(A',


In [58]:
predictions = [x.strip() for x in predictions]
predictions

['(C',
 '(A',
 '(C',
 '(A',
 '(C',
 '(B',
 '(C',
 '(A',
 '(B',
 '(C',
 '(C',
 '(A',
 '(C',
 '(C',
 '(C',
 '(C',
 '(A',
 '(C',
 '(A',
 '(B',
 '(B',
 '(A',
 '(B',
 '(C',
 '(C',
 '(A',
 '(A',
 '(B',
 '(A',
 '(A',
 '(A',
 '(A',
 '(A',
 '(C',
 '(B',
 '(A',
 '(A',
 '(A',
 '(A',
 '(C',
 '(C',
 '(A',
 '(A',
 '(A',
 '(A',
 '(A',
 '(C',
 '(B',
 '(A',
 '(A',
 '(C',
 '(E',
 '(A',
 '(C',
 '(B',
 '(C',
 '(C',
 '(A',
 '(A',
 '(A',
 '(A',
 '(A',
 '(E',
 '(C',
 '(C',
 '(A',
 '(C',
 '(C',
 '(A',
 '(A',
 '(C',
 '(A',
 '(A',
 '(A',
 '(A',
 '(C',
 '(A',
 '(C',
 '(A',
 '(A',
 '(C',
 '(C',
 '(C',
 '(A',
 '(C',
 '(C',
 '(C',
 '(C',
 '(C',
 '(B',
 '(A',
 '(A',
 '(A',
 '(A',
 '(C',
 '(A',
 '(C',
 '(C',
 '(B',
 '(A',
 '(C',
 '(C',
 '(C',
 '(C',
 '(B',
 '(B',
 '(A',
 '(A',
 '(C',
 '(B',
 '(A',
 '(E',
 '(C',
 '(C',
 '(D',
 '(A',
 '(B',
 '(A',
 '(D',
 '(A']

In [59]:
predictions = [x.strip("(").upper() for x in predictions]
predictions

['C',
 'A',
 'C',
 'A',
 'C',
 'B',
 'C',
 'A',
 'B',
 'C',
 'C',
 'A',
 'C',
 'C',
 'C',
 'C',
 'A',
 'C',
 'A',
 'B',
 'B',
 'A',
 'B',
 'C',
 'C',
 'A',
 'A',
 'B',
 'A',
 'A',
 'A',
 'A',
 'A',
 'C',
 'B',
 'A',
 'A',
 'A',
 'A',
 'C',
 'C',
 'A',
 'A',
 'A',
 'A',
 'A',
 'C',
 'B',
 'A',
 'A',
 'C',
 'E',
 'A',
 'C',
 'B',
 'C',
 'C',
 'A',
 'A',
 'A',
 'A',
 'A',
 'E',
 'C',
 'C',
 'A',
 'C',
 'C',
 'A',
 'A',
 'C',
 'A',
 'A',
 'A',
 'A',
 'C',
 'A',
 'C',
 'A',
 'A',
 'C',
 'C',
 'C',
 'A',
 'C',
 'C',
 'C',
 'C',
 'C',
 'B',
 'A',
 'A',
 'A',
 'A',
 'C',
 'A',
 'C',
 'C',
 'B',
 'A',
 'C',
 'C',
 'C',
 'C',
 'B',
 'B',
 'A',
 'A',
 'C',
 'B',
 'A',
 'E',
 'C',
 'C',
 'D',
 'A',
 'B',
 'A',
 'D',
 'A']

In [60]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.17      0.41      0.24        22
           B       0.40      0.24      0.30        25
           C       0.30      0.61      0.41        23
           D       0.50      0.04      0.08        24
           E       0.67      0.12      0.20        17
           F       0.00      0.00      0.00         7
           G       0.00      0.00      0.00         1
           H       0.00      0.00      0.00         1

    accuracy                           0.27       120
   macro avg       0.25      0.18      0.15       120
weighted avg       0.37      0.27      0.23       120



## Task: Boolean Expressions

In [61]:
dataset = load_json("/content/task_boolean_expressions.json")

In [62]:
dataset

,input,target
0,"From the following five expressions, only one ...",(E)
1,"From the following five expressions, only one ...",(C)
2,"From the following five expressions, only one ...",(E)
3,"From the following five expressions, only one ...",(D)
4,"From the following five expressions, only one ...",(C)
...,...,...
195,"From the following five expressions, only one ...",(A)
196,"From the following five expressions, only one ...",(B)
197,"From the following five expressions, only one ...",(B)
198,"From the following five expressions, only one ...",(C)


In [63]:
dataset["target"].value_counts()

,count
target,
(E),43
(B),41
(C),40
(A),39
(D),37


In [64]:
questions = dataset["input"].values.tolist()

In [65]:
true_answers = dataset["target"].values.tolist()

In [66]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'E', 'D', 'C', 'D', 'C', 'D', 'A', 'A']

In [67]:
instruction=f"""You are evaluating boolean expressions.

Exactly one option is True.

Output format rules:
- Output ONLY one of the following exact strings:
(A)
(B)
(C)
(D)
(E)

Do NOT use LaTeX.
Do NOT use special symbols.
Do NOT add explanations.
Do NOT add whitespace or extra characters.
"""

In [68]:
prompts = get_prompt(instruction,questions)

In [69]:
prompts[0]

'You are evaluating boolean expressions.\n\nExactly one option is True.\n\nOutput format rules:\n- Output ONLY one of the following exact strings:\n(A)\n(B)\n(C)\n(D)\n(E)\n\nDo NOT use LaTeX.\nDo NOT use special symbols.\nDo NOT add explanations.\nDo NOT add whitespace or extra characters.\n\n\nFrom the following five expressions, only one evaluates to true. Which one is it?\n\n(A) not (not (not (not (False) or (-5 - (1 / -5) is greater than -7)) or (not (False) or (-5 * -10 + -5 * -6 is greater than -4 * 6))) or (not (not (False) or (False)) or (not not not (-3 - (-2 / 4) is greater than -6) and not not (The capital of India is Mumbai.)))) and (not (not (not (The capital of the Nigeria is Abuja.) or (-7 - (-4 / 7) > 4)) or (not (-6 - (-7 / -5) > -6) and (True))) and (not not ((-1 * 9 is less than or equal to -6) or (-4 * 10 + -1 * -7 <= 9 * 10)) or not not not (not not (6 * -2 is less than or equal to 4) or not not not (-5 * 9 + 6 * -3 <= -9 * 6))))\n(B) ((not (not not not (not (not 

In [71]:
predictions = training(prompts,num_tokens=1)

Iteration 0

Response:  1
Iteration 1

Response:  1
Iteration 2

Response:  1
Iteration 3

Response:  1
Iteration 4

Response:  1
Iteration 5

Response:  1
Iteration 6

Response:  1
Iteration 7

Response:  1
Iteration 8

Response:  1
Iteration 9

Response:  1
Iteration 10

Response:  1
Iteration 11

Response:  1
Iteration 12

Response:  1
Iteration 13

Response:  1
Iteration 14

Response:  1
Iteration 15

Response:  1
Iteration 16

Response:  1
Iteration 17

Response:  1
Iteration 18

Response:  1
Iteration 19

Response:  1
Iteration 20

Response:  1
Iteration 21

Response:  1
Iteration 22

Response:  1
Iteration 23

Response:  1
Iteration 24

Response:  1
Iteration 25

Response:  1
Iteration 26

Response:  1
Iteration 27

Response:  1
Iteration 28

Response:  1
Iteration 29

Response:  1
Iteration 30

Response:  1
Iteration 31

Response:  1
Iteration 32

Response:  1
Iteration 33

Response:  1
Iteration 34

Response:  1
Iteration 35

Response:  1
Iteration 36

Response:  1
Iteration 3

In [72]:
predictions

[' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' ________________',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1',
 ' 1'

In [73]:
predictions = [x.strip() for x in predictions]
predictions

['1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '________________',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 

In [74]:
predictions = ['A' if x =='1' else 'Unknown' for x in predictions]
predictions

['A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'Unknown',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A',
 'A'

In [75]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.20      1.00      0.33        39
           B       0.00      0.00      0.00        41
           C       0.00      0.00      0.00        40
           D       0.00      0.00      0.00        37
           E       0.00      0.00      0.00        43
     Unknown       0.00      0.00      0.00         0

    accuracy                           0.20       200
   macro avg       0.03      0.17      0.05       200
weighted avg       0.04      0.20      0.06       200

